In [1]:
# ==== ZERO-SHOT TEST (WavLM, no fine-tuning) ====
# Baseline: Pretrained WavLM with a randomly initialized 2-class head

import os
import glob
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset
from transformers import AutoFeatureExtractor, WavLMForSequenceClassification, Trainer, TrainingArguments
import evaluate
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# --------------------
# Paths
# --------------------
CHECKPOINT = "microsoft/wavlm-base"
OUTPUT_DIR = "./wavlm-zero-shot"
TEST_DIR   = "Speech"  # CHANGE THIS to your test set path

# --------------------
# Dataset class (reuse)
# --------------------
class WavLMDataset(Dataset):
    def __init__(self, root_dir, feature_extractor, target_sr=16000, max_duration=10.0, recursive=True):
        self.files, self.labels = [], []
        self.feature_extractor = feature_extractor
        self.target_sr = target_sr
        self.max_length = int(target_sr * max_duration)

        # find wav files
        if recursive:
            candidates = glob.glob(os.path.join(root_dir, "**", "*.wav"), recursive=True)
            candidates += glob.glob(os.path.join(root_dir, "**", "*.WAV"), recursive=True)
        else:
            candidates = [os.path.join(root_dir, f) for f in os.listdir(root_dir) if f.lower().endswith(".wav")]

        for path in sorted(candidates):
            parent = os.path.basename(os.path.dirname(path)).lower()
            if parent == "real":
                label = 0
            elif parent == "fake":
                label = 1
            else:
                continue  # skip any files not inside Real/ or Fake folders

            self.files.append(path)
            self.labels.append(label)

        print(f"✅ Loaded {len(self.files)} files from {root_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, label = self.files[idx], self.labels[idx]
        waveform, sr = torchaudio.load(path)

        # mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        # resample
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)

        waveform = waveform.squeeze().numpy()

        # pad/truncate
        if len(waveform) > self.max_length:
            waveform = waveform[:self.max_length]
        else:
            pad = self.max_length - len(waveform)
            waveform = np.pad(waveform, (0, pad), mode="constant")

        # feature extractor
        inputs = self.feature_extractor(
            waveform,
            sampling_rate=self.target_sr,
            return_tensors="pt",
            padding=True
        )
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item


# --------------------
# Load extractor & zero-shot model
# --------------------
feature_extractor = AutoFeatureExtractor.from_pretrained(CHECKPOINT)

model = WavLMForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,                # binary head
    ignore_mismatched_sizes=True # replaces pretrained head
)

# --------------------
# Build test dataset
# --------------------
test_dataset = WavLMDataset(TEST_DIR, feature_extractor)

# --------------------
# Metrics
# --------------------
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
roc_metric = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "auroc": roc_metric.compute(prediction_scores=probs, references=labels)["roc_auc"]
    }

# --------------------
# Trainer (evaluate only)
# --------------------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_eval_batch_size=32,
    fp16=torch.cuda.is_available(),
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    eval_dataset=test_dataset,
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics
)

# --------------------
# Run evaluation
# --------------------
print("Evaluating zero-shot WavLM...")
metrics = trainer.evaluate(test_dataset)

print("\n=== Zero-Shot Test Metrics (WavLM) ===")
for k, v in metrics.items():
    if k.startswith("eval_"):
        print(f"{k[5:]}: {v:.4f}")

# --------------------
# Detailed classification report & confusion matrix
# --------------------
predictions = trainer.predict(test_dataset)
logits, labels = predictions.predictions, predictions.label_ids
y_pred = np.argmax(logits, axis=1)
probs_fake = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

target_names = ["Real (0)", "Fake (1)"]
rep_str = classification_report(labels, y_pred, target_names=target_names, digits=4)
print("\n=== Classification Report (Zero-Shot WavLM) ===")
print(rep_str)

cm = confusion_matrix(labels, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm,
    index=["True Real (0)", "True Fake (1)"],
    columns=["Pred Real (0)", "Pred Fake (1)"]
)
print("\n=== Confusion Matrix (Zero-Shot WavLM) ===")
print(cm_df)

# --------------------
# Save artifacts
# --------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(os.path.join(OUTPUT_DIR, "classification_report_test.txt"), "w", encoding="utf-8") as f:
    f.write(rep_str)

rep_df = pd.DataFrame(classification_report(labels, y_pred, target_names=target_names, output_dict=True)).transpose()
rep_df.to_csv(os.path.join(OUTPUT_DIR, "classification_report_test.csv"), index=True)

cm_df.to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_test.csv"), index=True)

per_file_df = pd.DataFrame({
    "Filename": [os.path.basename(p) for p in test_dataset.files],
    "TrueLabel": ["Real" if t == 0 else "Fake" for t in labels],
    "PredLabel": ["Real" if p == 0 else "Fake" for p in y_pred],
    "Prob_Fake": probs_fake
})
per_file_df.to_csv(os.path.join(OUTPUT_DIR, "per_file_predictions.csv"), index=False)

print(f"\n✅ Zero-shot WavLM results saved to {os.path.abspath(OUTPUT_DIR)}")


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Some weights of WavLMForSequenceClassification were not initialized from the model checkpoint at microsoft/wavlm-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Loaded 2544 files from Speech


C:\Users\j3n50\AppData\Local\Temp\ipykernel_11856\3309482889.py:131: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluating zero-shot WavLM...


c:\Users\j3n50\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(



=== Zero-Shot Test Metrics (WavLM) ===
loss: 0.6925
model_preparation_time: 0.0020
accuracy: 0.5016
f1_macro: 0.3340
auroc: 0.5549
runtime: 470.4249
samples_per_second: 5.4080
steps_per_second: 0.1700


c:\Users\j3n50\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(



=== Classification Report (Zero-Shot WavLM) ===
              precision    recall  f1-score   support

    Real (0)     0.5068    0.9800    0.6681      1302
    Fake (1)     0.0000    0.0000    0.0000      1242

    accuracy                         0.5016      2544
   macro avg     0.2534    0.4900    0.3340      2544
weighted avg     0.2594    0.5016    0.3419      2544


=== Confusion Matrix (Zero-Shot WavLM) ===
               Pred Real (0)  Pred Fake (1)
True Real (0)           1276             26
True Fake (1)           1242              0

✅ Zero-shot WavLM results saved to d:\Thesis\Song\ZershotSPeech\wavlm-zero-shot
